In [1]:
# 0. Erst die Bibliotheken installieren (falls noch nicht geschehen)
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.6 MB/s eta 0:00:00


In [3]:
import torch
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# --- 1. Basic Configuration ---
model_id = "unsloth/llama-3-8b-instruct-bnb-4bit"
output_dir = "./llama3-tax-standard"

# --- 2. Load Resources Independently ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# --- 3. PEFT Setup (LoRA) ---
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# --- 4. Robust Tokenization Function ---
# Instead of relying on the Trainer to format, we tokenize the data ourselves.
# This avoids all "dataset_text_field" or "max_seq_length" TypeErrors.
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=1024 # Conservative length to save memory
    )

# Load the master CSV you prepared
raw_dataset = load_dataset("csv", data_files="train_dataset_final.csv", split="train")

# Map the tokenization across the dataset
tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# --- 5. Standard Training Pipeline ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_steps=60,
    logging_steps=1,
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none",
    remove_unused_columns=False
)

# Use the standard Trainer with a Data Collator
# This is the most stable combination in the Hugging Face ecosystem
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Starting standard training pipeline...")
trainer.train()
print("Training completed successfully.")

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5440 [00:00<?, ? examples/s]

Starting standard training pipeline...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.842841
2,1.693866
3,1.822488
4,1.705675
5,1.662952
6,1.584250
7,1.487292
8,1.500214
9,1.636271
10,1.629907


Training completed successfully.


In [7]:
import pandas as pd
import torch

# --- 1. Inference Optimization ---
# Set the model to evaluation mode to disable dropout and other training-specific behaviors
model.eval()

# Configure tokenizer for batch inference:
# Left-padding is mandatory for causal LLMs like Llama-3 during batch generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 2. Data Loading ---
# Load the cleaned test dataset containing the legal cases
df_test = pd.read_csv("dataset_clean.csv")
results = []

# --- 3. High-Performance Batch Inference ---
# Batching increases GPU utilization and significantly reduces total execution time
BATCH_SIZE = 16
print(f"Starting optimized batch inference for {len(df_test)} cases (Batch Size: {BATCH_SIZE})...")

for i in range(0, len(df_test), BATCH_SIZE):
    # Extract the current batch of data
    batch_df = df_test.iloc[i : i + BATCH_SIZE]

    # Construct prompts using the fine-tuned instruction format
    batch_prompts = [
        f"### Instruction:\nAnalysiere den folgenden steuerrechtlichen Sachverhalt und nenne die relevanten österreichischen Paragraphen.\n\n### Input:\n{row['prompt']}\n\n### Response:\n"
        for _, row in batch_df.iterrows()
    ]

    # Tokenize the batch with padding and truncation to ensure uniform tensor shapes
    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to("cuda")

    # Execute inference using torch.inference_mode for reduced memory overhead and faster computation
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,      # Constrain generation length as legal citations are concise
            use_cache=True,          # Enable KV-caching for faster decoding
            do_sample=False,         # Use greedy decoding for deterministic and reproducible legal output
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode batch outputs and remove special tokens
    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Parse generated responses and map them back to their original IDs
    for idx, full_text in enumerate(decoded_outputs):
        # Extract content following the '### Response:' marker
        if "### Response:\n" in full_text:
            answer = full_text.split("### Response:\n")[-1].strip()
        else:
            answer = full_text.strip()

        results.append({
            "id": batch_df.iloc[idx]['id'],
            "answer": answer
        })

    # Console feedback for monitoring processing status
    current_progress = min(i + BATCH_SIZE, len(df_test))
    completion_rate = (current_progress / len(df_test)) * 100
    print(f"Progress: {current_progress}/643 ({completion_rate:.1f}%)")

# --- 4. Result Export ---
# Save the model predictions to a CSV file for final evaluation
output_df = pd.DataFrame(results)
output_df.to_csv("model_2_results_final.csv", index=False)
print("\nInference completed. Results exported to 'model_2_results_final.csv'.")

Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting optimized batch inference for 643 cases (Batch Size: 16)...


Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Progress: 16/643 (2.5%)


Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Progress: 608/643 (94.6%)


Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Progress: 624/643 (97.0%)


Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Progress: 640/643 (99.5%)
Progress: 643/643 (100.0%)

Inference completed. Results exported to 'model_2_results_final.csv'.
